In [1]:
import os
if not os.path.exists("week10_students.csv"):
    from google.colab import files
    uploaded = files.upload()

Saving week10_students.xlsx to week10_students.xlsx


In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, confusion_matrix

df = pd.read_excel("week10_students.xlsx")
df.head()

,student_id,name,gender,program,district,attendance_pct,quiz_avg,assignments_avg,score
0,AUCA001,Aline Uwase,F,Information Systems,Gasabo,55,31.0,35.0,28.6
1,AUCA002,Eric Iradukunda,M,Software Engineering,Musanze,73,61.1,76.7,64.5
2,AUCA003,Diane Uwizeyimana,F,Networking,Muhanga,92,35.2,65.5,52.7
3,AUCA004,Jean Bosco Ikirezi,M,Accounting,Kicukiro,69,65.2,54.2,68.3
4,AUCA005,Patrick Ingabire,M,Information Systems,Huye,88,39.3,42.9,42.3


In [4]:
print("Shape:", df.shape)
features = ["quiz_avg", "assignments_avg", "attendance_pct"]
print(df[features + ["score"]].corr()["score"].drop("score"))


Shape: (150, 9)
quiz_avg           0.800726
assignments_avg    0.535983
attendance_pct     0.238118
Name: score, dtype: float64


In [5]:
X = df[features]
y = df["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

print("Train:", len(X_train), "| Test:", len(X_test))

Train: 112 | Test: 38


In [6]:
reg = LinearRegression()
reg.fit(X_train, y_train)

for name, coef in zip(X.columns, reg.coef_):
    print(f"{name:18s} {coef:8.3f}")
print("Intercept:", round(reg.intercept_, 3))

quiz_avg              0.579
assignments_avg       0.365
attendance_pct        0.113
Intercept: -4.428


In [7]:
pred = reg.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2  = r2_score(y_test, pred)
print(f"MAE: {mae:.2f} points")     # checkpoint ~3.3
print(f"R2 : {r2:.3f}")

MAE: 3.33 points
R2 : 0.887


In [8]:
new_student = pd.DataFrame({"quiz_avg": [70], "assignments_avg": [75], "attendance_pct": [90]})
print("Predicted score:", round(reg.predict(new_student)[0], 1))

# Your own (invented numbers - change them!)
me = pd.DataFrame({"quiz_avg": [65], "assignments_avg": [70], "attendance_pct": [85]})
print("My predicted score:", round(reg.predict(me)[0], 1))

Predicted score: 73.7
My predicted score: 68.4


In [9]:
df["passed"] = (df["score"] >= 50).astype(int)
print(df["passed"].value_counts())

yc = df["passed"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, yc, test_size=0.25, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(Xc_train, yc_train)

passed
1    131
0     19
Name: count, dtype: int64


LogisticRegression(max_iter=1000)

In [10]:
pred_c = clf.predict(Xc_test)

print("Model accuracy :", round(accuracy_score(yc_test, pred_c), 3))
print("Lazy baseline  :", round(yc_test.mean(), 3))   # predict "pass" for everyone
print("Confusion matrix (rows = actual [fail, pass], cols = predicted [fail, pass]):")
print(confusion_matrix(yc_test, pred_c))

Model accuracy : 0.895
Lazy baseline  : 0.895
Confusion matrix (rows = actual [fail, pass], cols = predicted [fail, pass]):
[[ 3  1]
 [ 3 31]]


In [11]:
def test_r2(Xdata):
    Xtr, Xte, ytr, yte = train_test_split(Xdata, y, test_size=0.25, random_state=42)
    m = LinearRegression().fit(Xtr, ytr)
    return r2_score(yte, m.predict(Xte))

X_quiz = df[["quiz_avg"]]
X_three = df[features]
X_prog = pd.get_dummies(df[features + ["program"]], columns=["program"], dtype=int)

results = pd.DataFrame({
    "Version": ["1. quiz_avg only", "2. three numeric features", "3. three + program (one-hot)"],
    "Test R2": [test_r2(X_quiz), test_r2(X_three), test_r2(X_prog)]})
print(results.round(3).to_string(index=False))

                     Version  Test R2
            1. quiz_avg only    0.624
   2. three numeric features    0.887
3. three + program (one-hot)    0.880


In [12]:
fig1 = px.histogram(df, x="score", nbins=12, title="1. Where the class stands")
fig1.show()

df["status"] = df["passed"].map({1: "Pass", 0: "At risk"})   # readable legend
fig2 = px.scatter(df, x="quiz_avg", y="score", color="status", hover_name="name",
                  title="2. Who is at risk")
fig2.show()

district_avg = df.groupby("district", as_index=False)["score"].mean()
fig3 = px.bar(district_avg, x="district", y="score", title="3. Where to act")
fig3.show()

In [13]:
fig2.write_html("class_dashboard.html")
print("Saved class_dashboard.html - run the next cell to download it, then open it in your browser, hover around, and screenshot it.")

Saved class_dashboard.html - run the next cell to download it, then open it in your browser, hover around, and screenshot it.


In [14]:
# Colab: download the dashboard to your computer
from google.colab import files
files.download("class_dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
fig4 = px.box(df, x="program", y="score", color="program",
              title="Score distribution by program")
fig4.show()